# DWBI Week 3 — Fact & Dimension Tables
**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A**

PakFreight's flat `Shipments_Unified` (34 rows, 578,650 kg) becomes **one fact table + five dimensions** — and the totals do not move by a single kilogram.

## T6 — Build FactShipment

Left of the gap: `Shipments_Unified` (grey, untouched). Right of the gap: the fact table —
**six integer keys, one degenerate dimension, three measures, no descriptive text**:

`DateKey · OriginCityKey · DestCityKey · RouteKey · ModeKey · StatusKey · ShipmentID · WeightKG · FreightCostPKR · ShipmentCount`

Row 6 was worked; the lookups were copied down (until T4/T5 existed, every key read `-1` —
the Unknown row doing its job, not an error).

**Q1 — what `IFERROR(…, -1)` buys and costs:** it buys *no vanishing rows* — unrecognised context lands
on Unknown, visible and countable (T7 check 12 watches it). It costs *silence* — a broken mapping looks
like data, so you must monitor the -1 count, never trust it blindly.

**Q2 — why text doesn't cross the gap:** the dimension owns descriptive text — stored once, defined once.
Text in the fact would duplicate per row, rot the conformed definition, and collapse the star back into
the flat file Week 2 escaped.

**Q3 — why ShipmentID stays:** the degenerate dimension — the operational reference users ask about
("where is MR-1001?"). It joins to nothing because there is no Shipment table; deleting it would destroy
traceability to the source.

In [1]:
import pandas as pd
f = pd.read_csv("data/FactShipment.csv")
text_cols = [c for c in f.columns if f[c].dtype == object and c != "ShipmentID"]
print("rows:", len(f), "| keys all integer:", all(str(f[c].dtype).startswith("int") for c in
      ["DateKey","OriginCityKey","DestCityKey","RouteKey","ModeKey","StatusKey"]))
print("descriptive text columns in fact:", text_cols, "(only the degenerate ShipmentID)")
print("totals:", f"{f.WeightKG.sum():,.0f} kg,", f"{f.FreightCostPKR.sum():,.0f} PKR")

rows: 34 | keys all integer: True
descriptive text columns in fact: [] (only the degenerate ShipmentID)
totals: 578,650 kg, 4,808,000 PKR
